# ELECTRA + ScalarMix + DANN — DeepSeek synthetic train → OOD eval



- Train: DeepSeek synthetic Q/A from Google Drive
- Test: OneStop English + RACE middle/high from Hugging Face (**original labels**)
- Saves model, metrics, and confusion matrices to `DRIVE_OUT_DIR`

**Domains:** The synthetic CSV has one source (DeepSeek). We assign **hash-based shard domains** (8 buckets over `question+answer`) so the domain head has something to adversarially confuse. This regularizes representations; it is not corpus-level DANN. For true multi-corpus DANN, train on merged multi-source CSVs with a `source_dataset` column instead.

In [ ]:
!pip install -q transformers datasets scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths & hyperparameters ──────────────────────────────────────────────
DRIVE_SYNTH_CSV = "/content/drive/MyDrive/multi_corpus/synthetic_deepseek_qa.csv"
DRIVE_OUT_DIR = "/content/drive/MyDrive/beyond_flesch/trail/electra_deepseek_dann"

MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 512
BATCH_SIZE = 4
VAL_FRAC = 0.20
RNG_SEED = 42
LABEL_SMOOTHING = 0.05

# DANN
NUM_DOMAINS = 8
GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1

# Two-phase training (match gentle DANN recipe)
PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 2
PHASE2_LR = 2e-5
PARTIAL_FREEZE_LAYERS = 8

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}
LABEL_NAMES = ["elementary", "middle", "high"]

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import Tensor
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
(Path(DRIVE_OUT_DIR) / "confusion_matrices").mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)
print("Output:", DRIVE_OUT_DIR)

In [ ]:
# Shared model definitions live in DomainAdversialNN/dann_models.py
import sys
from pathlib import Path

for module_dir in [
    Path.cwd(),
    Path.cwd() / "DomainAdversialNN",
    Path("/content/drive/MyDrive/BeyondFlesch/Beyond-Flesch/DomainAdversialNN"),
    Path("/content/drive/MyDrive/BeyondFK/DomainAdversialNN"),
]:
    if module_dir.exists() and str(module_dir) not in sys.path:
        sys.path.append(str(module_dir))

from dann_models import (
    ElectraScalarMixDANN,
    combined_loss,
    grl_lambda_schedule,
)

print("Shared DANN model imports OK")

In [ ]:
def shard_domain_id(text: str, n_domains: int = NUM_DOMAINS) -> int:
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(h, 16) % n_domains


# ── Load DeepSeek synthetic CSV ────────────────────────────────────────────
if not os.path.exists(DRIVE_SYNTH_CSV):
    alt = os.path.basename(DRIVE_SYNTH_CSV)
    if os.path.exists(alt):
        DRIVE_SYNTH_CSV = alt
    else:
        raise FileNotFoundError(f"Synthetic CSV not found: {DRIVE_SYNTH_CSV}")

synth = pd.read_csv(DRIVE_SYNTH_CSV)
synth.columns = [c.strip().lower() for c in synth.columns]
for col in ("question", "answer", "grade_level"):
    if col not in synth.columns:
        raise ValueError(f"Expected column '{col}' in {DRIVE_SYNTH_CSV}, got {list(synth.columns)}")

synth = synth.dropna(subset=["question", "answer", "grade_level"]).reset_index(drop=True)
synth["grade_level"] = synth["grade_level"].astype(str).str.strip().str.lower()
synth = synth[synth["grade_level"].isin(label2id)].reset_index(drop=True)
synth["full_text"] = (
    "Question: " + synth["question"].astype(str) + "\n"
    "Answer: " + synth["answer"].astype(str)
)
synth["label"] = synth["grade_level"].map(label2id).astype(int)
synth["domain"] = synth["full_text"].map(shard_domain_id)

print(f"Synthetic rows: {len(synth)}")
print(synth["grade_level"].value_counts())
print("Domain shard counts:", synth["domain"].value_counts().sort_index().to_dict())

X = synth["full_text"].tolist()
y = synth["label"].values
d = synth["domain"].values
X_tr, X_va, y_tr, y_va, d_tr, d_va = train_test_split(
    X, y, d, test_size=VAL_FRAC, stratify=y, random_state=RNG_SEED
)
print(f"Train {len(X_tr)} | Val {len(X_va)}")

In [ ]:
# ── OOD corpora from Hugging Face (original labels) ──────────────────────

def load_onestop_hf() -> Tuple[List[str], np.ndarray]:
    ds = load_dataset("SetFit/onestop_english")
    texts, labels = [], []
    mapping = {0: 0, 1: 1, 2: 2}
    for split in ds:
        for r in ds[split]:
            li = int(r.get("label", -1))
            if li not in mapping:
                continue
            text = (r.get("text") or r.get("content") or "").strip()
            if text:
                texts.append(text)
                labels.append(mapping[li])
    return texts, np.array(labels, dtype=int)


def load_race_hf(config: str) -> Tuple[List[str], np.ndarray]:
    assert config in ("middle", "high")
    ds = load_dataset("ehovy/race", config)
    level_id = label2id[config]
    texts, labels, seen = [], [], set()
    for split in ds:
        for r in ds[split]:
            article = (r.get("article") or "").strip()
            if not article or article in seen:
                continue
            seen.add(article)
            texts.append(article)
            labels.append(level_id)
    return texts, np.array(labels, dtype=int)


print("Loading OneStop English …")
OSE_X, OSE_y = load_onestop_hf()
print(f"  OneStop: n={len(OSE_X)}  class counts={np.bincount(OSE_y, minlength=3)}")

print("Loading RACE middle …")
RACE_M_X, RACE_M_y = load_race_hf("middle")
print(f"  RACE-middle: n={len(RACE_M_X)}")

print("Loading RACE high …")
RACE_H_X, RACE_H_y = load_race_hf("high")
print(f"  RACE-high: n={len(RACE_H_X)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDANNDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, domains: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)
        self.domains = domains.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


def make_loader(texts, labels, domains, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextDANNDataset(texts, labels, domains),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = make_loader(X_tr, y_tr, d_tr, shuffle=True)
val_loader = make_loader(X_va, y_va, d_va, shuffle=False)

In [ ]:
# ── Train: Phase 1 (frozen encoder, heads only) → Phase 2 (DANN + partial unfreeze) ──
model = ElectraScalarMixDANN(MODEL_NAME, num_domains=NUM_DOMAINS).to(device)
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")
history: List[Dict] = []
best_val_f1 = -1.0
global_step = 0
total_steps = (PHASE1_EPOCHS + PHASE2_EPOCHS) * len(train_loader)


@torch.no_grad()
def evaluate_val(grl_lambda: float = 0.0) -> Dict[str, float]:
    model.eval()
    ys, preds, ds, dp = [], [], [], []
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_lambda)
        ys.extend(batch["label"].numpy().tolist())
        preds.extend(diff_logits.argmax(dim=1).cpu().numpy().tolist())
        ds.extend(batch["domain"].numpy().tolist())
        dp.extend(dom_logits.argmax(dim=1).cpu().numpy().tolist())
    return {
        "val_macro_f1": float(f1_score(ys, preds, average="macro", zero_division=0)),
        "val_acc": float(accuracy_score(ys, preds)),
        "val_domain_acc": float(accuracy_score(ds, dp)),
    }


def run_phase(phase_name: str, epochs: int, lr: float, grl_on: bool) -> None:
    global global_step, best_val_f1
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=min(50, max(1, len(train_loader))),
        num_training_steps=max(1, epochs * len(train_loader)),
    )
    for epoch in range(epochs):
        model.train()
        run_diff, run_dom, run_total, n_batches = 0.0, 0.0, 0.0, 0
        for batch in tqdm(train_loader, desc=f"{phase_name} ep{epoch+1}/{epochs}"):
            progress = global_step / max(total_steps - 1, 1)
            grl_l = grl_lambda_schedule(progress, max_lambda=GRL_LAMBDA_MAX) if grl_on else 0.0
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            domains = batch["domain"].to(device)
            diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_l)
            total, diff_loss, dom_loss = combined_loss(
                diff_logits,
                dom_logits,
                labels,
                domains,
                domain_loss_alpha=DOMAIN_LOSS_ALPHA,
                label_smoothing=LABEL_SMOOTHING,
            )
            optimizer.zero_grad()
            total.backward()
            optimizer.step()
            scheduler.step()
            global_step += 1
            run_diff += diff_loss.item()
            run_dom += dom_loss.item()
            run_total += total.item()
            n_batches += 1
        metrics = evaluate_val(grl_lambda=0.0)
        rec = {
            "phase": phase_name,
            "epoch": epoch + 1,
            "grl_on": grl_on,
            "grl_lambda": grl_l if grl_on else 0.0,
            "train_diff_loss": run_diff / max(n_batches, 1),
            "train_dom_loss": run_dom / max(n_batches, 1),
            "train_total_loss": run_total / max(n_batches, 1),
            **metrics,
        }
        history.append(rec)
        print(
            f"{phase_name} ep{epoch+1}: diff={rec['train_diff_loss']:.4f} dom={rec['train_dom_loss']:.4f} "
            f"val_f1={rec['val_macro_f1']:.4f} val_dom_acc={rec['val_domain_acc']:.4f}"
        )
        if metrics["val_macro_f1"] > best_val_f1:
            best_val_f1 = metrics["val_macro_f1"]
            torch.save(model.state_dict(), best_path)
            print(f"  → saved best (val macro-F1 {best_val_f1:.4f})")


# Phase 1: encoder frozen
for p in model.encoder.parameters():
    p.requires_grad = False
run_phase("phase1_frozen_encoder", PHASE1_EPOCHS, PHASE1_LR, grl_on=False)

# Phase 2: partial unfreeze + DANN
model.unfreeze_encoder()
model.freeze_encoder_except_top(PARTIAL_FREEZE_LAYERS)
run_phase("phase2_dann", PHASE2_EPOCHS, PHASE2_LR, grl_on=True)

model.load_state_dict(torch.load(best_path, map_location=device))
print("Training done. Best val macro-F1:", best_val_f1)

In [ ]:
# ── OOD eval + confusion matrices (difficulty head only, grl=0) ───────────
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")


def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(
            texts[i : i + batch_size],
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        )
        logits = model.difficulty_logits_only(
            enc["input_ids"].to(device), enc["attention_mask"].to(device)
        )
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_corpus(name: str, texts: List[str], y_true: np.ndarray, labels: List[int]) -> Dict:
    y_pred = predict_texts(texts)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=labels,
        target_names=[id2label[i] for i in labels], zero_division=0,
    )
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(report)
    print(f"Accuracy: {acc:.4f}  Macro-F1: {f1m:.4f}")
    cm_path = os.path.join(CM_DIR, f"cm_{name.replace(' ', '_').lower()}.png")
    save_confusion_matrix(y_true, y_pred, name, cm_path, labels)
    print(f"Saved CM → {cm_path}")
    return {
        "corpus": name, "n": int(len(y_true)),
        "accuracy": float(acc), "macro_f1": float(f1m),
        "labels_evaluated": labels, "classification_report": report,
        "confusion_matrix_png": cm_path,
    }


results = {
    "config": {
        "model": MODEL_NAME,
        "dann": True,
        "synth_csv": DRIVE_SYNTH_CSV,
        "num_domains": NUM_DOMAINS,
        "domain_sharding": "md5_hash_mod",
        "grl_lambda_max": GRL_LAMBDA_MAX,
        "domain_loss_alpha": DOMAIN_LOSS_ALPHA,
        "phase1_epochs": PHASE1_EPOCHS,
        "phase2_epochs": PHASE2_EPOCHS,
        "best_val_macro_f1": float(best_val_f1),
        "train_history": history,
    },
    "evaluations": [],
}

for name, texts, y_true, labels in [
    ("synthetic_val", X_va, y_va, [0, 1, 2]),
    ("onestop_english", OSE_X, OSE_y, [0, 1, 2]),
    ("race_middle", RACE_M_X, RACE_M_y, [0, 1, 2]),
    ("race_high", RACE_H_X, RACE_H_y, [0, 1, 2]),
]:
    results["evaluations"].append(eval_corpus(name, texts, y_true, labels))

summary = pd.DataFrame([
    {"corpus": e["corpus"], "n": e["n"], "accuracy": e["accuracy"], "macro_f1": e["macro_f1"]}
    for e in results["evaluations"]
])
print("\nSummary:\n", summary.to_string(index=False))

json_path = os.path.join(DRIVE_OUT_DIR, "eval_results.json")
csv_path = os.path.join(DRIVE_OUT_DIR, "eval_summary.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
summary.to_csv(csv_path, index=False)
print(f"\nSaved JSON → {json_path}")
print(f"Saved CSV  → {csv_path}")
print(f"Model      → {best_path}")
print(f"CMs        → {CM_DIR}")